# 10 · Few-shot for Catalan: compare on TUNE, then ONE run on TEST (dev log 09)

Runtime → Change runtime type → **GPU (T4)**.

**Rule decided before running:** keep `--fewshot` only if TUNE *CO₂e OK* improves (tie → fewer confident errors). Only then run it once on TEST.

In [1]:
!git clone -q -b llm-matcher https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt transformers accelerate bitsandbytes

/content/Food-Sustainability


## 1. TUNE only (TEST is not touched)

In [2]:
!python src/evaluate_matching.py --matcher llm --model Qwen/Qwen2.5-14B-Instruct --verify --tune-only

config.json: 100% 663/663 [00:00<00:00, 3.77MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 42.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 117MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 130MB/s]
model.safetensors.index.json: 100% 47.5k/47.5k [00:00<00:00, 58.5MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.89G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.88G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.5G [00:00<?, ?B/s]
Reconstructing (incomplete tota

In [4]:
!python src/evaluate_matching.py --matcher llm --model Qwen/Qwen2.5-14B-Instruct --verify --fewshot --tune-only

Loading weights: 100% 579/579 [02:00<00:00,  4.81it/s]
[W1120 12:24:38.391640222 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 465567744 bytes (free: 341639168, total: 15637086208).
Matcher: llm (Qwen/Qwen2.5-14B-Instruct + verify + fewshot) | TUNE ONLY, TEST not touched
Threshold: 0.3
TUNE: 33/50 correct, 41/50 CO2e OK, 9 uncertain, 9 confident errors
          correct  co2e_ok   n
language                      
ca              5        7  12
en             17       22  25
es             11       12  13


In [5]:
import sys, re
sys.path.insert(0, "src")
from evaluate_matching import load_labels, predict, apply_threshold, build_matcher
from llm_matching import FEWSHOT_EXAMPLES

examples = set(re.findall(r'- "([^"]+)"', FEWSHOT_EXAMPLES))
labels = load_labels()
tune44 = labels[(labels.split == "tune") & ~labels.ingredient_text.isin(examples)]

for fs in [False, True]:
    m, _ = build_matcher("llm", model="Qwen/Qwen2.5-14B-Instruct", provider="hf", verify=True, fewshot=fs)
    r = apply_threshold(predict(m, tune44), 0.3)
    print("fewshot" if fs else "base   ", f"{len(r)} words | correct {r.correct.sum()} | CO2e OK {r.co2e_ok.sum()} "
          f"| uncertain {r.uncertain.sum()} | confident errors {r.confident_error.sum()}")
    print(r.groupby("language")[["correct", "co2e_ok"]].sum().T, "\n")

base    44 words | correct 32 | CO2e OK 38 | uncertain 5 | confident errors 8
language  ca  en  es
correct    5  17  10
co2e_ok    6  21  11 

fewshot 44 words | correct 32 | CO2e OK 39 | uncertain 5 | confident errors 8
language  ca  en  es
correct    5  17  10
co2e_ok    6  22  11 



## 2. Only if fewshot won on TUNE: ONE run on TEST, then recompute the app's AI matches

Copy the TUNE numbers into `docs/09_dev_log_ai_in_the_app.md` first.

In [6]:
FEWSHOT_WON = True   # <- set to True only if it won on TUNE
if FEWSHOT_WON:
    !python src/evaluate_matching.py --matcher llm --model Qwen/Qwen2.5-14B-Instruct --verify --fewshot --details
    !python src/precompute_matches.py --model Qwen/Qwen2.5-14B-Instruct --verify --fewshot

Loading weights: 100% 579/579 [02:00<00:00,  4.79it/s]
Matcher: llm (Qwen/Qwen2.5-14B-Instruct, retrieve top-20 + choose + verify)
Objective: accuracy
Threshold chosen on TUNE: 0.3  (tune accuracy 66%)

TEST: 30/50 correct (60%)
      9 uncertain (no number), 13 confident errors (wrong food with a CO2e number)
      CO2e OK (right food, or CO2e within +-25%): 38/50
          correct  co2e_ok   n
language                      
ca              5        6  13
en             17       19  23
es              8       13  14
 ingredient_text                                  agribalyse_name                                        pred_name  score  uncertain  correct
            beef                 Beef, minced steak, 15% fat, raw                                   Beef, rib, raw    1.0      False    False
      black bean                    Haricot bean, canned, drained                                Haricot bean, dry    0.0       True    False
      burger bun     Rolls for hamburger/hotdog (bu

## 3. Download and commit

`data/results_llm_*_test.csv`, `data/ingredient_matches_llm.csv` and `data/llm_cache.json` (Files panel → right click → Download).

In [8]:
from google.colab import files
for f in ["data/ingredient_matches_llm.csv", "data/llm_cache.json",
          "data/results_llm_Qwen2.5-14B-Instruct_verify_fewshot_test.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>